# Day-to-day evolution of (ridepooling) supply and demand
Module for simulating ridesourcing evolution, including pooled rides

Contribution by Arjan de Ruijter - a.j.f.deruijter@tudelft.nl

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys # add MaaSSim and MaaSSim/MaaSSim to path (not needed if already in path)
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from MaaSSim.utils import save_config, get_config, load_G, generate_demand, initialize_df, empty_series, \
    slice_space, test_space, read_requests_csv
from MaaSSim.maassim import Simulator
from MaaSSim.data_structures import structures as inData
from MaaSSim.d2d_sim import *
from MaaSSim.d2d_demand import *
from MaaSSim.d2d_supply import *
from MaaSSim.d2d_shared import prep_shared_rides
from MaaSSim.decisions import dummy_False
from MaaSSim.exmas import main

In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)
import zipfile
import logging
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
import numpy as np
import random
# import ExMAS
plt.style.use('ggplot')
np.random.seed(0)
random.seed(0)

In [4]:
# Load config
params = get_config('../../data/config/ams.json')  # load configuration
params.paths.albatross = '../../data/albatross'

In [5]:
# Experiment replications and number of threads to be used
params.parallel.nReplications = 1
params.parallel.nThread = 1

# Main experimental settings
params.nP = 1000 # travellers
params.nV = 25 # drivers
params.nD = 3 # days
params.simTime = 2 # hours

In [6]:
# Other day-to-day settings
params.evol.drivers.kappa = 0.2 # learning weight (supply-side)
params.evol.drivers.res_wage.mean = 25 #euros/h
params.evol.drivers.gini = 0.35 # gini coefficient used to establish sigma parameter of log-norm distribution of res wage
params.evol.drivers.init_inc_ratio = 1 #expected income of informed drivers at start of sim as ratio of res wage

params.evol.drivers.inform.prob_start = 1 # probability of being informed at start of sim
params.evol.drivers.inform.beta = 0.1 # information transmission rate
params.evol.drivers.inform.std_fact = 0.5 # multiplier of the standard deviation of experienced income used in signal

params.evol.drivers.regist.prob_start = 1 # probability of being registered if informed at start of sim
params.evol.drivers.regist.beta = 0.2 # registration choice model parameter
params.evol.drivers.regist.cost_comp = 20 # daily share of registration costs (euros)
params.evol.drivers.regist.samp = 0.5 # probability of making (de)regist decision
params.evol.drivers.regist.min_work_exp = 0 # Working experience required before deregistration is possible
params.evol.drivers.regist.min_days = 5 # Minimum number of registered days before driver can deregister

params.evol.drivers.particip.beta = 0.1 # participation choice model parameter
params.evol.drivers.particip.probabilistic = True # stochasticity in participation choice

params.evol.travellers.inform.prob_start = 1 # probability that traveller is informed at start of sim
params.evol.travellers.inform.beta = 0.1 # information transmission rate (demand-side)
params.evol.travellers.inform.start_wait = 0 # expected waiting time at start of simulation
params.evol.travellers.inform.std_fact = 0.5 # multiplier of the standard deviation of experienced waiting time used in signal
params.evol.travellers.reject_penalty = 30 * 60 # seconds
params.evol.travellers.kappa = 0.2 # learning weight (demand-side)
params.evol.travellers.min_prob = 0.05 # filtering criterion, when probability is lower when waiting time is zero, never consider RS

params.evol.travellers.mode_pref.mean_vot = 10 # Mean VoT in euro/h
params.evol.travellers.mode_pref.access_multip = 2 # Multiplier of access time compared to in-vehicle time
params.evol.travellers.mode_pref.wait_multip = 2.5 # Multiplier of waiting time compared to in-vehicle time
params.evol.travellers.mode_pref.bike_multip = 2 # Multiplier of biking time compared to in-vehicle time
params.evol.travellers.mode_pref.beta_cost = -0.364 #-0.296 # -0.148 # util/euro #-0.1592
params.evol.travellers.mode_pref.transfer_pen = 5 * 60 # seconds, to be added to IVT for each transfer
params.evol.travellers.mode_pref.ASC_car = -1.683 #-0.159 # -1.216 # util, rel to bike
params.evol.travellers.mode_pref.ASC_rs = -4.599 #-2.419 #-3.172
params.evol.travellers.mode_pref.ASC_pt = -3.724 #-1.782 #-2.303
params.evol.travellers.mode_pref.ASC_car_sd = 2.647 #3.338 # standard deviation in ASCs
params.evol.travellers.mode_pref.ASC_rs_sd = 1.682 #2.068 #1
params.evol.travellers.mode_pref.ASC_pt_sd = 1.355 #2.341 #1
params.evol.travellers.mode_pref.ASC_bike_sd = 5.570 #0 #1
params.evol.travellers.mode_pref.gini = params.evol.drivers.gini

# Financial settings
params.platforms.base_fare = 1.5 #euro
params.platforms.fare = 1.5 #euro/km
params.platforms.min_fare = 0 # euro
params.platforms.comm_rate = 0.25 #rate
params.drivers.fuel_costs = 0.25 #euro/km

# Properties alternative modes
params.alt_modes.pt.option = False  # PT alternatives are generated based on Albatross dataset, so Albatross dataset needs to be loaded
params.alt_modes.pt.base_fare = 1 # euro
params.alt_modes.pt.km_fare = 0.2 # euro/km
params.alt_modes.car.km_cost = 0.5 # euro/km
params.alt_modes.car.diff_parking = True # different parking tariffs in city
params.alt_modes.car.park_cost = 7.5 # euro
params.alt_modes.car.park_cost_center = 15 # euro
params.alt_modes.car.access_time = 10 * 60 # s
params.speeds.bike = (1/2.5) * params.speeds.ride # m/s

# Regulation
params.platforms.reg_cap = np.inf # registration cap
params.platforms.ptcp_cap = np.inf # daily participation cap

# Demand settings
params.demand.albatross = False # do we take demand from Albatross dataset
# params.demand_structure.origins_dispertion = -0.0003
# params.demand_structure.destinations_dispertion = -0.0003
params.dist_threshold_min = 2000 # min dist
# params.dist_threshold = 100000 # max dist

# Start time
# params.t0 = pd.Timestamp.now()
params.t0 = pd.Timestamp(2021, 11, 1, 9)

In [7]:
# Pooling settings
params.shareability.offered = True
params.shareability.min_discount = 0.25 # fraction of solo fare
params.shareability.add_discount = 0.25 # fraction of solo fare
params.shareability.shared_discount = params.shareability.min_discount # as considered by platform in determ. of feasible rides (fraction of solo fare), currently not used
params.shareability.delay_value = 1 # currently not used
params.shareability.WtS_mean = 1.5 # Actual (mean) WtS multipl. in population
params.shareability.WtS = 1 # Willingness to share multipl. as considered by platform in determining feasible rides
params.shareability.sharing_constant = -0.251 #-0.215 #-0.215 #util, rel. to private ride
params.shareability.sharing_const_sd = 0 #0.065
params.shareability.WtS_std = 0 # St. dev. of WtS in population
params.shareability.matching_obj = 'u_pax' #minimize VHT for vehicles
params.shareability.pax_delay = 0 # additional delay for picking up travs
params.shareability.max_delay = 10 * 60 # max. allowed delay in pooled rides (seconds)
params.shareability.horizon = 600
params.shareability.max_degree = 3
params.shareability.share = 1
params.shareability.without_matching = True

In [8]:
# Compute (other) input parameters in ExMAS
params.shareability.avg_speed = params.speeds.ride
params.shareability.price = params.platforms.fare #eur/km
params.shareability.nP = params.nP
params.shareability.VoT = params.evol.travellers.mode_pref.mean_vot / 3600 # convert eur/h to eur/s   # not used?

In [9]:
inData = load_G(inData, params, stats=True, set_t=False)  # download graph for the 'params.city' and calc the skim matrices
if params.alt_modes.car.diff_parking:
    inData = diff_parking(inData) # determine which nodes are in center

In [10]:
# inData = generate_demand(inData, params, avg_speed = True)

In [11]:
if params.demand.albatross:
    inData = load_albatross_proc(inData, params, avg_speed = True)  # Load processed Albatross file
    inData.requests = inData.requests.drop(['orig_geo', 'dest_geo', 'origin_y', 'origin_x', 'destination_y', 'destination_x', 'time'], axis = 1)
#     inData.requests.insert(0, 'pax_id', inData.requests.pop('pax_id'))
    if params.alt_modes.pt.option: # Do we allow for a PT option - trip option taken from OpenTripPlanner?
        inData.pt_itinerary = load_OTP_result(params)
        inData = select_reqs_with_OTP_offer(inData)
    inData = sample_from_alba(inData, params)
else:
    inData = generate_demand(inData, params, avg_speed = True)

In [12]:
# if params.alt_modes.pt.option: # Do we allow for a PT option - trip option taken from OpenTripPlanner?
#     # Load processed Albatross file, the OTP result, and compute PT fares
#     inData = load_albatross_proc(inData, params, avg_speed = True)
#     inData.requests = inData.requests.drop(['orig_geo', 'dest_geo', 'origin_y', 'origin_x', 'destination_y', 'destination_x', 'time'], axis = 1)
#     inData.requests.insert(0, 'pax_id', inData.requests.pop('pax_id'))
#     inData.pt_itinerary = load_OTP_result(params)
#     inData = consist_OTP_alba(inData, params)
# else:
#     inData = generate_demand(inData, params, avg_speed = True)

In [13]:
# Prepare supply and demand attributes
inData.passengers = prefs_travs(inData, params) # Draw mode preferences and determine utilities of alternative modes
all_pax = mode_filter(inData, params) # Determine probability to choose ride-hailing without waiting time and delay
inData.passengers = all_pax[all_pax.mode_choice == "day-to-day"] # Select those with significant probability to choose ride-hailing
inData.requests = inData.requests[inData.requests.pax_id.isin(inData.passengers.index)] # Select requests accordingly
if params.alt_modes.pt.option:
    inData.pt_itinerary = inData.pt_itinerary[inData.pt_itinerary.pax_id.isin(inData.passengers.index)]
    inData.pt_itinerary.reset_index(drop=True, inplace=True)
inData.passengers.reset_index(drop=True, inplace=True)
inData.requests.reset_index(drop=True, inplace=True)
inData.requests['pax_id'] = inData.requests.index
inData.pt_itinerary['pax_id'] = inData.pt_itinerary.index
inData.passengers['informed'] = np.random.rand(len(inData.passengers)) < params.evol.travellers.inform.prob_start
inData.passengers['expected_wait'] = params.evol.travellers.inform.start_wait
inData.passengers['expected_wait_pool'] = params.evol.travellers.inform.start_wait
inData.passengers['expected_pool_disc'] = params.shareability.min_discount
inData.passengers['expected_pool_detour'] = 0
fixed_supply = generate_vehicles_d2d(inData, params) # generate vehicle data
inData.vehicles = fixed_supply.copy()
inData.vehicles.platform = inData.vehicles.apply(lambda x: 0, axis = 1)
inData.passengers.platforms = inData.passengers.apply(lambda x: [0], axis = 1)
inData.requests['platform'] = inData.requests.apply(lambda row: inData.passengers.loc[row.name].platforms[0], axis = 1) 
inData.platforms = pd.concat([inData.platforms,pd.DataFrame(columns=['base_fare','comm_rate','min_fare'])])
inData.platforms = initialize_df(inData.platforms)
inData.platforms.loc[0]=[params.platforms.fare,'Uber',30,params.platforms.base_fare,params.platforms.comm_rate,params.platforms.min_fare,]

In [14]:
inData = main(inData, params.shareability, plot=False) # create shareability graph (ExMAS), i.e. set of feasible pooled rides 

04-06-23 19:03:27-INFO-Initializing pairwise trip shareability between 1000 and 1000 trips.
04-06-23 19:03:27-INFO-creating combinations
04-06-23 19:03:27-INFO-138012	 nR*(nR-1)
04-06-23 19:03:31-INFO-Reduction of feasible pairs by 99.59%
04-06-23 19:03:31-INFO-Degree 2 	Completed
04-06-23 19:03:32-INFO-trips to extend at degree 2 : 9420
04-06-23 19:04:23-INFO-At degree 2 feasible extensions found out of 30026 searched
04-06-23 19:04:23-INFO-Degree 3 	Completed
04-06-23 19:04:23-INFO-Max degree reached 3
04-06-23 19:04:23-INFO-Trips still possible to extend at degree 3 : 30026


In [15]:
# Day-to-day simulation (incl. processing)
sim = Simulator(inData, params=params,
                    kpi_veh = D2D_veh_exp,
                    kpi_pax = d2d_kpi_pax,
                    f_driver_out = D2D_driver_out,
                    f_trav_out = d2d_no_request,
                    f_trav_mode = dummy_False,
                    logger_level=logging.WARNING)  # initialize

evol_micro = init_d2d_dotmap(params)
plf_stats = pd.DataFrame(columns = ['profit','veh_dist','pax_dist'])
for day in range(params.get('nD', 1)):  # run iterations
    inData.passengers = mode_preday(inData, params)
    temp_rides = inData.sblts.rides.copy()
    temp_reqs = inData.sblts.requests.copy()
    
    rs_users = inData.passengers[(inData.passengers.mode_day == 'rs') | (inData.passengers.mode_day == 'pool')].index.tolist()
    poolers = inData.passengers[inData.passengers.mode_day == 'pool'].index.tolist()
    inData.sblts.rides = inData.sblts.rides[inData.sblts.rides.apply(lambda x: all(i in rs_users for i in x.indexes), axis=1)] # filter out all travellers opting for mode outside ride-hailing market
    inData.sblts.rides = inData.sblts.rides[inData.sblts.rides.apply(lambda x: (all(i in poolers for i in x.indexes) or x.kind == 1), axis=1)]  # filter out pooled option for individuals opting for private ride
    inData.sblts.requests = inData.sblts.requests[inData.sblts.requests.apply(lambda x: x.pax_id in rs_users, axis=1)]
    
    inData = prep_shared_rides(inData, params.shareability)  # prepare schedules
    sim.make_and_run(run_id=day)  # prepare and SIM
    sim.output()  # calc results
    sim.last_res = sim.res[day].copy()
    del sim.res[day]

    drivers_summary = update_d2d_drivers(sim=sim,params=params)
    travs_summary = update_d2d_travellers(sim=sim,params=params)
    
    exp_df = update_work_exp(inData, drivers_summary)
    inData.vehicles.work_exp = exp_df.work_exp
    inData.days_since_reg = exp_df.days_since_reg

    res_inf_driver = wom_driver(inData, params = params)
    inData.vehicles.informed = res_inf_driver
    inData.vehicles.expected_income = learning_unregist(inData, drivers_summary, params = params)
    
    res_regist = platform_regist(inData, drivers_summary, params = params)
    inData.vehicles.registered = res_regist.registered
    inData.vehicles.work_exp = res_regist.work_exp
    inData.vehicles.pos = fixed_supply.pos
    inData.vehicles.rejected_reg = res_regist.rejected_reg
    
    res_inf_trav = wom_trav(inData, travs_summary, params = params)
    inData.passengers.informed = res_inf_trav.informed
    inData.passengers.expected_wait = res_inf_trav.perc_wait
    inData.passengers.expected_wait_pool = res_inf_trav.perc_wait_pool
    inData.passengers.expected_pool_disc = res_inf_trav.perc_disc
    inData.passengers.expected_pool_detour = res_inf_trav.perc_detour
    
    inData.sblts.rides = temp_rides.copy()
    inData.sblts.requests = temp_reqs.copy()
    
    evol_micro = d2d_summary_day(evol_micro, drivers_summary, travs_summary, day)
    evol_micro = d2d_summ_pooling(evol_micro, travs_summary)
    row = {'profit': sim.last_res.veh_kpi.REVENUE.loc['sum'] / (1 - params.platforms.comm_rate), 'veh_dist': sim.last_res.veh_kpi.DRIVING_DIST.loc['sum'], 'pax_dist': sim.last_res.pax_kpi.TRAVEL.loc['sum'] * params.speeds.ride / 1000}
    plf_stats = plf_stats.append(row, ignore_index=True)

04-06-23 18:49:49-WARNING-Setting up 2h simulation at 2021-11-01 09:01:24 for 25 vehicles and 1000 passengers in Amsterdam, Netherlands


KeyError: 175

In [ ]:
evol_micro, evol_agg = d2d_agg_statistics(evol_micro, params) # multi-day stats

In [ ]:
# Save d2d stats to zip file
with zipfile.ZipFile('evol.zip', 'w') as csv_zip:
    csv_zip.writestr("evol_agg_supply.csv", evol_agg.supply.to_csv())
    csv_zip.writestr("evol_agg_demand.csv", evol_agg.demand.to_csv())

In [ ]:
evol_agg.supply

In [ ]:
evol_agg.demand

In [ ]:
# Plot evolution of main performance indicators
fig, axes = plt.subplots(nrows=9, ncols=1, figsize = (10,20), sharex = True)
evol_agg.supply[['inform','regist','particip']].plot(ax = axes[0], color=['lightsteelblue','tab:blue','midnightblue'])
axes[0].set_title('(A) Ridesourcing supply')
axes[0].legend(['Informed','Registered','Participating'])
axes[0].set_ylim([0,params.nV + 25])
axes[0].set_ylabel('Number of drivers')
evol_agg.supply[['mean_perc_inc','mean_exp_inc']].plot(ax = axes[2], color=['lightsteelblue','midnightblue'])
axes[2].set_title('(C) Driver earnings')
axes[2].legend(['Expected','Experienced'])
axes[2].set_ylim([0,math.ceil(max(evol_agg.supply.mean_perc_inc.max(),evol_agg.supply.mean_exp_inc.max())/50)*50])
axes[2].set_ylabel('Income (\u20ac)')

evol_agg.demand[['req_solo','req_pool','bike','car','pt']].plot.area(ax = axes[1])
h,l = axes[1].get_legend_handles_labels()
evol_agg.demand['inform'].plot(ax = axes[1], color = 'black', linestyle = 'dashed', label = 'informed')
line = Line2D([0], [0],color='black', linestyle ='dashed')
axes[1].set_title('(B) Demand')
axes[1].set_ylim([0,len(inData.passengers) * 1.1])
axes[1].set_ylabel('Number of travellers')
h.extend([line])
axes[1].legend(labels=["Solo RS","Pooling RS","Bike","Car","Public transport","Informed"], handles=h)

evol_agg.demand['mean_wait_solo'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Experienced - solo')
# evol_agg.demand['corr_mean_wait_solo'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Corrected exp. - solo')
evol_agg.demand['perc_wait'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Expected - solo')
evol_agg.demand['mean_wait_pooling'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Experienced - pooled')
# evol_agg.demand['corr_mean_wait_pooling'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Corrected exp. - pooled')
evol_agg.demand['perc_wait_pool'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Expected - pooled')
axes[3].legend()
axes[3].set_title('(D) Waiting time')
axes[3].set_ylabel('Mean wait. time (min)')

evol_agg.demand['proport_match_solo'] = evol_agg.demand.gets_offer_solo / evol_agg.demand.req_solo * 100
evol_agg.demand['proport_match_pool'] = evol_agg.demand.gets_offer_pooling / evol_agg.demand.req_pool * 100
evol_agg.demand['act_shared_rel'] = evol_agg.demand.act_shared / evol_agg.demand.gets_offer_pooling * 100
evol_agg.demand['proport_match_solo'].plot(ax = axes[4], label='Solo requests returned with offer')
evol_agg.demand['proport_match_pool'].plot(ax = axes[4], label='Pooled requests returned with offer')
evol_agg.demand['act_shared_rel'].plot(ax = axes[4], label='Share of pooled rides with actual sharing', linestyle = 'dotted')
axes[4].legend()
axes[4].set_title('(E) Proportion matched')
axes[4].yaxis.set_major_formatter(mtick.PercentFormatter())

evol_agg.demand['mean_detour'].apply(lambda x: 1/60 * x).plot(ax = axes[5], label='Experienced detour')
evol_agg.demand['perc_detour'].apply(lambda x: 1/60 * x).plot(ax = axes[5], label='Expected detour')
axes[5].legend()
axes[5].set_title('(F) Pooling detour')
axes[5].set_ylabel('Mean detour time (min)')

evol_agg.demand['mean_disc'].apply(lambda x: 100 * x).plot(ax = axes[6], label='Experienced discount')
evol_agg.demand['perc_disc'].apply(lambda x: 100 * x).plot(ax = axes[6], label='Expected discount')
axes[6].legend()
axes[6].set_title('(G) Pooling discount')
axes[6].yaxis.set_major_formatter(mtick.PercentFormatter())

plf_stats['profit'].plot(ax = axes[7], label='Platform profit')
axes[7].set_title('(H) Platform profit')

plf_stats['dist_ratio'] = plf_stats.pax_dist / plf_stats.veh_dist
plf_stats['dist_ratio'].plot(ax = axes[8], label='Pax km per veh km')
axes[8].legend()
axes[8].set_title('(I) Distance savings')

plt.savefig('d2d-evo.png')
plt.show()

---

In [ ]:
sim.last_res.veh_exp

In [ ]:
sim.last_res.pax_exp

In [ ]:
inData.sblts.R[1]

In [ ]:
inData.sblts.rides

In [ ]:
inData.sblts.requests

In [ ]:
sim.runs[0].rides

In [ ]:
sim.runs[0].trips

In [ ]:
inData.sblts.SINGLES

In [ ]:
xyz = inData.sblts.rides[inData.sblts.rides.apply(lambda x: all(i in rs_users for i in x.indexes), axis=1)] # filter out all travellers opting for mode outside ride-hailing market
xyz = xyz[xyz.apply(lambda x: (all(i in poolers for i in x.indexes) or x.kind == 1), axis=1)]  # filter out pooled trips for individuals opting for private ride
xyz
# poolers

In [ ]:
xyz = inData.sblts.schedule.copy()
xyz['pooling_reqs'] = xyz.apply(lambda x: any(i in travs_summary[travs_summary.chosen_mode == 'pool'].index.to_list() for i in x.indexes) and travs_summary.gets_offer.loc[x.indexes[0]], axis=1)
xyz[xyz.pooling_reqs]

In [ ]:
inData.passengers.head(50)

In [ ]:
travs_summary

In [ ]:
all_pax.head(50)

In [ ]:
all_pax[all_pax['mode_choice'] != 'day-to-day'].mode_choice.value_counts()

In [ ]:
all_pax['prob_rs_tot'] = all_pax.prob_rs + all_pax.prob_pool

In [ ]:
all_pax.prob_rs_tot.hist(bins=100)
plt.show()

In [ ]:
inData.sblts.rides

In [17]:
inData.sblts.requests.head(50)

TypeError: 'DotMap' object is not callable

In [ ]:
inData.sblts.rides.tail(50)

In [16]:
inData.requests

,origin,destination,treq,tarr,ttrav,dist,haver_dist,ttrav_alb,schedule_id,shareable,tdrop,pax_id,tdep,car_park_cost,dest_center,platform
0,6932152175,1069290042,2021-11-01 09:02:50,1900-01-01 09:20:00,0 days 00:05:51,3510,2535.088979,0 days 00:18:00,2,False,NaN,0,NaN,15.0,True,0
1,46399459,46434407,2021-11-01 09:04:35,1900-01-01 09:25:00,0 days 00:07:55,4759,4181.166746,0 days 00:21:00,3,False,NaN,1,NaN,7.5,False,0
2,46327150,46365575,2021-11-01 09:05:04,1900-01-01 09:17:00,0 days 00:06:17,3772,2496.937899,0 days 00:12:00,4,False,NaN,2,NaN,7.5,False,0
3,287776238,46307582,2021-11-01 09:05:08,1900-01-01 09:25:00,0 days 00:07:07,4278,3200.739744,0 days 00:20:00,5,False,NaN,3,NaN,7.5,False,0
4,46385765,46566460,2021-11-01 09:05:09,1900-01-01 09:56:00,0 days 00:17:52,10721,6020.331560,0 days 00:51:00,6,False,NaN,4,NaN,7.5,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380,46352910,359038020,2021-11-01 17:56:00,1900-01-01 18:01:00,0 days 00:05:18,3185,2112.674343,0 days 00:05:00,989,False,NaN,380,NaN,7.5,False,0
381,297879416,46246551,2021-11-01 17:56:52,1900-01-01 18:01:00,0 days 00:07:49,4691,2439.131271,0 days 00:05:00,993,False,NaN,381,NaN,7.5,False,0
382,6376341837,46308872,2021-11-01 17:57:55,1900-01-01 18:01:00,0 days 00:05:01,3012,2269.236614,0 days 00:04:00,995,False,NaN,382,NaN,7.5,False,0
383,7230055451,46386760,2021-11-01 17:58:34,1900-01-01 18:15:00,0 days 00:12:48,7682,5875.843018,0 days 00:17:00,997,False,NaN,383,NaN,15.0,True,0


In [ ]:
inData.sblts.keys()

In [ ]:
inData.sblts.requests

In [ ]:
poolers

In [15]:
inData.requests.sort_values(by=['treq'])

,origin,destination,treq,tarr,ttrav,dist,haver_dist,ttrav_alb,tdrop,schedule_id,pax_id,shareable,tdep
4339493,271391422,46202475,2021-11-01 09:00:20,1900-01-01 09:35:00,0 days 00:10:55,6550,4959.698421,0 days 00:35:00,NaN,4339493,4339493,False,NaN
4166679,1906771030,46504086,2021-11-01 09:01:24,1900-01-01 09:18:00,0 days 00:07:55,4750,2975.073698,0 days 00:17:00,NaN,4166679,4166679,False,NaN
4420485,6932152175,1069290042,2021-11-01 09:02:50,1900-01-01 09:20:00,0 days 00:05:51,3510,2535.088979,0 days 00:18:00,NaN,4420485,4420485,False,NaN
4205814,46399459,46434407,2021-11-01 09:04:35,1900-01-01 09:25:00,0 days 00:07:55,4759,4181.166746,0 days 00:21:00,NaN,4205814,4205814,False,NaN
4256857,46327150,46365575,2021-11-01 09:05:04,1900-01-01 09:17:00,0 days 00:06:17,3772,2496.937899,0 days 00:12:00,NaN,4256857,4256857,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4230230,6376341837,46308872,2021-11-01 17:57:55,1900-01-01 18:01:00,0 days 00:05:01,3012,2269.236614,0 days 00:04:00,NaN,4230230,4230230,False,NaN
4412664,46402528,46288517,2021-11-01 17:57:59,1900-01-01 18:13:00,0 days 00:06:42,4022,2924.538271,0 days 00:16:00,NaN,4412664,4412664,False,NaN
4389507,7230055451,46386760,2021-11-01 17:58:34,1900-01-01 18:15:00,0 days 00:12:48,7682,5875.843018,0 days 00:17:00,NaN,4389507,4389507,False,NaN
4263121,940042043,46327288,2021-11-01 17:58:38,1900-01-01 18:09:00,0 days 00:05:12,3127,2018.837295,0 days 00:11:00,NaN,4263121,4263121,False,NaN


In [15]:
inData.sblts.requests.head(50)

,index,pax_id,origin,destination,treq,tdep,ttrav,tarr,tdrop,shareable,schedule_id,dist,car_park_cost,dest_center,platform,VoT,delta,u,u_PT
0,0,0,252315496,46380158,0,NaN,805,2021-11-01 08:13:35,NaN,False,NaN,8050,15.0,True,0,0.002778,600.000,14.311111,999999
1,1,1,4422115072,46351183,5,NaN,649,2021-11-01 08:11:04,NaN,False,NaN,6498,15.0,True,0,0.002778,600.000,11.549778,999999
2,2,2,46257961,46380093,24,NaN,378,2021-11-01 08:06:52,NaN,False,NaN,3789,15.0,True,0,0.002778,511.515,6.733500,999999
3,3,3,46332982,46314424,63,NaN,345,2021-11-01 08:06:58,NaN,False,NaN,3455,7.5,False,0,0.002778,466.425,6.140833,999999
4,4,4,46380158,46424347,78,NaN,206,2021-11-01 08:04:54,NaN,False,NaN,2065,15.0,True,0,0.002778,278.775,3.669722,999999
5,5,5,46411016,46430144,121,NaN,641,2021-11-01 08:12:52,NaN,False,NaN,6412,15.0,True,0,0.002778,600.000,11.398556,999999
6,6,6,5656789643,1069290042,144,NaN,529,2021-11-01 08:11:23,NaN,False,NaN,5297,15.0,True,0,0.002778,600.000,9.414944,999999
7,7,7,46286182,5897198108,169,NaN,431,2021-11-01 08:10:10,NaN,False,NaN,4316,7.5,False,0,0.002778,582.660,7.671222,999999
8,8,8,46247649,46343363,172,NaN,308,2021-11-01 08:08:10,NaN,False,NaN,3081,15.0,True,0,0.002778,415.935,5.477056,999999
9,9,9,46428748,46390250,226,NaN,207,2021-11-01 08:07:23,NaN,False,NaN,2070,15.0,True,0,0.002778,279.450,3.680000,999999


In [ ]:
inData.passengers.to_csv('pass_off.csv',index=False)

In [ ]:
inData.requests.to_csv('reqs_off.csv',index=False)

In [17]:
inData.pt_itinerary

DotMap(index=DotMap(), pax_id=DotMap(...), _ipython_display_=DotMap(), _repr_mimebundle_=DotMap())

In [28]:
abc=inData.requests

In [30]:
abc.sort_values(by='treq')
# abc.reset_index(inplace=True, drop=True)

,origin,destination,treq,tarr,ttrav,dist,haver_dist,ttrav_alb,pax_id,tdrop,shareable,tdep,schedule_id,car_park_cost,dest_center,platform
181,1906771030,46504086,2021-11-01 09:01:24,1900-01-01 09:18:00,0 days 00:07:55,4750,2975.073698,0 days 00:17:00,181,NaN,True,NaN,454,7.5,False,0
134,46327150,46365575,2021-11-01 09:05:04,1900-01-01 09:17:00,0 days 00:06:17,3772,2496.937899,0 days 00:12:00,134,NaN,False,NaN,336,7.5,False,0
236,46385765,46566460,2021-11-01 09:05:09,1900-01-01 09:56:00,0 days 00:17:52,10721,6020.331560,0 days 00:51:00,236,NaN,False,NaN,632,7.5,False,0
249,46414075,46363812,2021-11-01 09:05:20,1900-01-01 09:19:00,0 days 00:05:03,3034,2049.164051,0 days 00:14:00,249,NaN,False,NaN,668,7.5,False,0
61,4934571916,46411017,2021-11-01 09:05:47,1900-01-01 09:36:00,0 days 00:15:27,9277,4649.279646,0 days 00:31:00,61,NaN,False,NaN,144,7.5,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,2004730870,46404530,2021-11-01 17:54:47,1900-01-01 18:01:00,0 days 00:04:46,2867,2198.934112,0 days 00:07:00,220,NaN,False,NaN,579,7.5,False,0
65,46402284,5615352271,2021-11-01 17:55:39,1900-01-01 18:01:00,0 days 00:05:09,3099,2273.699426,0 days 00:06:00,65,NaN,False,NaN,149,7.5,False,0
126,46349210,46296689,2021-11-01 17:56:26,1900-01-01 18:13:00,0 days 00:06:53,4131,3578.912444,0 days 00:17:00,126,NaN,False,NaN,316,7.5,False,0
128,46317938,46316356,2021-11-01 17:57:21,1900-01-01 18:05:00,0 days 00:07:31,4517,3004.785406,0 days 00:08:00,128,NaN,True,NaN,321,7.5,False,0


In [31]:
abc

,origin,destination,treq,tarr,ttrav,dist,haver_dist,ttrav_alb,pax_id,tdrop,shareable,tdep,schedule_id,car_park_cost,dest_center,platform
0,46374662,6864381165,2021-11-01 09:38:50,1900-01-01 10:00:00,0 days 00:15:55,9552,6213.221697,0 days 00:22:00,0,NaN,False,NaN,2,7.5,False,0
1,46337326,46392233,2021-11-01 17:31:59,1900-01-01 17:49:00,0 days 00:11:39,6995,5567.811322,0 days 00:18:00,1,NaN,False,NaN,3,7.5,False,0
2,46439291,1624052865,2021-11-01 15:56:21,1900-01-01 16:01:00,0 days 00:07:00,4207,3378.544163,0 days 00:05:00,2,NaN,False,NaN,4,7.5,False,0
3,442122363,46487431,2021-11-01 14:48:50,1900-01-01 15:11:00,0 days 00:08:18,4980,3089.071140,0 days 00:23:00,3,NaN,False,NaN,5,7.5,False,0
4,296703862,46314951,2021-11-01 11:56:25,1900-01-01 12:11:00,0 days 00:04:37,2775,2195.527825,0 days 00:15:00,4,NaN,False,NaN,8,7.5,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,46388735,46317329,2021-11-01 13:55:38,1900-01-01 14:18:00,0 days 00:05:17,3175,2217.793862,0 days 00:23:00,382,NaN,False,NaN,989,7.5,False,0
383,46436685,46440061,2021-11-01 09:42:57,1900-01-01 10:01:00,0 days 00:05:50,3502,2823.942397,0 days 00:19:00,383,NaN,False,NaN,993,7.5,False,0
384,46303605,46247913,2021-11-01 12:28:12,1900-01-01 12:37:00,0 days 00:06:24,3840,2641.114852,0 days 00:09:00,384,NaN,False,NaN,995,7.5,False,0
385,3185290420,46277919,2021-11-01 16:23:08,1900-01-01 16:40:00,0 days 00:06:01,3612,2889.156965,0 days 00:17:00,385,NaN,False,NaN,997,7.5,False,0


In [ ]:
abc = abc.sort_index()